TOOLS 

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool

@tool
def get_weather(city:str)->str:
    """"get the weather of the city"""
    return f"the weather of {city} is sunny"


In [9]:
model=init_chat_model("gemini-2.5-flash",model_provider="google_genai")
model_binding=model.bind_tools([get_weather])
output=model_binding.invoke("what is the weather of ooty?")
print(output)

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "ooty"}'}, '__gemini_function_call_thought_signatures__': {'d3e122bc-28c6-4cda-bab0-c3795accab85': 'CqMCAQw51sc4065v6WnCDm2QFEUTLVMBY72d3pRDzZGVyrqsElEuSIeMIrR93EKEJ27MSXYsy1JK4kAY27ocgEXNRVZ31HNPg1vpN/c889ybGbjQYPx96ehrU592Zs8tO0Mxcy2cbSlgefQXzUGpFpFW9uVmKTGwgBmh4smtDgYeV91kqeHrocF6uuvdLm3sjuNsK9pkzpdSVSKcrAxfcH/uX903CFNhFhJyAvxjjsT9NMdzdpdTKd+7FCe1xvpGDeBeXAZscIFK3wERWeJf5lj0vzxCmRC1btSsy0KJKanoyJ6vFdW20gZG5O/x+3Uy1Pk8ZNgaswLXzo6O3zthGjkiDqEEvTxdxkbVqI0xu+od6dYmdz4sduWN/6habk6gSjoE42A8'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019ec447-0ac8-76f2-9e23-2fe3db680580-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'ooty'}, 'id': 'd3e122bc-28c6-4cda-bab0-c3795accab85', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 85, 'total_tokens':

In [10]:
for tool_call in output.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'city': 'ooty'}


Tool Execution Loops

In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_binding.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_binding.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [12]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'7910d6a4-ef99-41d6-9233-55e0fe791538': 'CuUBAQw51seIhZ7CXa/s8r4ZbkSI6dSec1786w9caJXvFo6SZ/Ji0kp8HBBEDw4qWM0EYSZU18VX47FD8PObXNjSBwIu5VQ6S6Uq6k9V8cYwAak09AT0ixRU+7MlsrgPEyVvnqIvM9Rjbu371ZCDpYVNv/LwYSf5SA5BiadTc742JEG5gg+PZ2jr+7FEFlZvMMvgHSVcGSAj6Vw4ekI/9S66kVmbHvRdMsEAWYrbGp39E3mr2HVvokwou3HQ1CDn+BPepBseQ6rw+GTksveajYl1e8qg4ZCmle/rNc3ipmdcK3sh2Vb9Xw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec4cd-ae35-7b52-8b00-8a29ccdd0b87-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': '7910d6a4-ef99-41d6-9233-55e0fe791538', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 62, 'total_toke